# Scoring Sensitivity and Internal Consistency Analysis

> **IMPORTANT FRAMING**
>
> This notebook does NOT validate business outcomes.
> No sales, volume, or profit labels exist yet.
>
> **Purpose:**
> 1. Check if manual scoring weights behave consistently across stores and categories
> 2. Identify which features drive score variance most
> 3. Verify rule-based scoring is internally consistent
> 4. Sensitivity analysis: if a weight changes slightly, how much do scores change?
>
> **This is NOT:**
> - Proof that scores predict real business outcomes
> - ML validation of business logic
> - A replacement for outcome-based validation (planned when 6+ months of data accumulates)
>
> **Use in README:**
> *"Scoring sensitivity analysis to verify manual opportunity weights behave consistently
> across stores and categories — not business outcome validation, which requires sales data."*

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from config import PROJECT_ID, CREDENTIALS_PATH
from google.oauth2 import service_account
from google.cloud import bigquery

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from scipy.stats import spearmanr

pd.set_option('display.float_format', '{:.2f}'.format)
print("Imports OK")

In [ ]:
creds = service_account.Credentials.from_service_account_file(
    CREDENTIALS_PATH, scopes=['https://www.googleapis.com/auth/bigquery'])
client = bigquery.Client(project=PROJECT_ID, credentials=creds)

sql = '''
SELECT
    aq.store_id,
    aq.category_name,
    aq.overall_opportunity_score,
    aq.opportunity_tier,
    aq.overall_demand_gap_score,
    er.expansion_readiness_score,
    aq.overall_confidence_score,
    aq.overall_risk_score,
    aq.overall_margin_risk_score,
    aq.pricing_power_score,
    aq.markdown_safety_score,
    pi.price_gap_pct,
    pi.adjusted_price_gap_pct
FROM `windy-container-451804-n4.bronze.mart_action_queue` aq
LEFT JOIN `windy-container-451804-n4.bronze.mart_expansion_readiness` er
    ON aq.store_id = er.store_id AND aq.category_name = er.category_name
LEFT JOIN `windy-container-451804-n4.bronze.mart_pricing_intelligence` pi
    ON aq.store_id = pi.store_id AND aq.category_name = pi.category_name
'''
df = client.query(sql).to_dataframe()
print(f"Shape: {df.shape}")
print(f"Stores: {df['store_id'].nunique()}  Categories: {df['category_name'].nunique()}")
print(f"Missing values:\n{df.isnull().sum()[df.isnull().sum()>0]}")
df = df.fillna(df.median(numeric_only=True))
print(f"\nAfter fillna — shape: {df.shape}")

In [ ]:
# Store-level holdout split
# NOT temporal — only 1 month of data, temporal split not possible
print("Store-level holdout — NOT temporal split")
print("Temporal split not possible with 1 month of data")
print("Revalidate after 6+ months accumulates")
print()

np.random.seed(42)
all_stores = df['store_id'].unique()
test_stores  = np.random.choice(all_stores, size=5, replace=False)
train_stores = [s for s in all_stores if s not in test_stores]

train_df = df[df['store_id'].isin(train_stores)].copy()
test_df  = df[df['store_id'].isin(test_stores)].copy()

print(f"Train stores ({len(train_stores)}): {sorted(train_stores)}")
print(f"Test  stores ({len(test_stores)}):  {sorted(test_stores)}")
print(f"Train rows: {len(train_df)}  Test rows: {len(test_df)}")

## Section 1 — Score Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Tier distribution
tier_counts = df['opportunity_tier'].value_counts()
tier_order  = ['Prime','Solid','Watch','Low']
tier_colors = {'Prime':'#10B981','Solid':'#3B82F6','Watch':'#F59E0B','Low':'#6B7280'}
ordered = [tier_counts.get(t, 0) for t in tier_order]
bars = axes[0].bar(tier_order, ordered,
                   color=[tier_colors[t] for t in tier_order], alpha=0.85)
axes[0].set_title('Opportunity Tier Distribution', fontsize=12)
axes[0].set_ylabel('Count')
for bar, val in zip(bars, ordered):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.3, str(val),
                 ha='center', va='bottom', fontsize=10)

# Score histogram
axes[1].hist(df['overall_opportunity_score'], bins=20, color='#3B82F6', alpha=0.8, edgecolor='white')
axes[1].axvline(df['overall_opportunity_score'].mean(), color='red', linestyle='--',
                label=f"Mean={df['overall_opportunity_score'].mean():.1f}")
axes[1].axvline(df['overall_opportunity_score'].median(), color='orange', linestyle=':',
                label=f"Median={df['overall_opportunity_score'].median():.1f}")
axes[1].set_title('Overall Opportunity Score Distribution', fontsize=12)
axes[1].set_xlabel('Score (0–100)')
axes[1].legend()

# Score std check
score_std = df['overall_opportunity_score'].std()
flag = "FLAG: Low variance" if score_std < 5 else "OK: Adequate spread"
axes[1].set_ylabel(f'Count  ({flag})')

# Box plots by category
cats_sorted = sorted(df['category_name'].unique())
score_by_cat = [df[df['category_name']==c]['overall_opportunity_score'].values for c in cats_sorted]
bp = axes[2].boxplot(score_by_cat, labels=cats_sorted, patch_artist=True,
                     medianprops=dict(color='red', lw=2))
for patch, color in zip(bp['boxes'], plt.cm.tab10(np.linspace(0,1,len(cats_sorted)))):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[2].set_xticklabels(cats_sorted, rotation=45, ha='right', fontsize=8)
axes[2].set_title('Score Distribution by Category', fontsize=12)
axes[2].set_ylabel('Opportunity Score')

plt.tight_layout()
os.makedirs('../docs/screenshots', exist_ok=True)
plt.savefig('../docs/screenshots/score_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Score stats — Mean: {df['overall_opportunity_score'].mean():.1f}  "
      f"Std: {score_std:.1f}  Min: {df['overall_opportunity_score'].min():.1f}  "
      f"Max: {df['overall_opportunity_score'].max():.1f}")
print(f"Low variance flag: {'YES' if score_std < 5 else 'NO'}  (std={score_std:.1f})")

## Section 2 — Feature Correlation Analysis

In [ ]:
feature_cols = [
    'overall_demand_gap_score', 'expansion_readiness_score',
    'overall_confidence_score', 'overall_risk_score',
    'overall_margin_risk_score', 'pricing_power_score',
    'markdown_safety_score', 'price_gap_pct'
]
feature_cols = [c for c in feature_cols if c in df.columns]

corr_with_score = df[feature_cols + ['overall_opportunity_score']].corr()['overall_opportunity_score'].drop('overall_opportunity_score')
top5 = corr_with_score.abs().sort_values(ascending=False).head(5)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
corr_mat = df[feature_cols].corr()
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, ax=axes[0], mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1, square=True,
            xticklabels=[c.replace('_score','').replace('overall_','') for c in feature_cols],
            yticklabels=[c.replace('_score','').replace('overall_','') for c in feature_cols])
axes[0].set_title('Feature Correlation Matrix', fontsize=12)
axes[0].tick_params(labelsize=8)

# Correlation with target
bar_colors = ['#10B981' if v >= 0 else '#EF4444' for v in corr_with_score]
axes[1].barh(
    [c.replace('_score','').replace('overall_','') for c in corr_with_score.index],
    corr_with_score.values, color=bar_colors, alpha=0.85)
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_title('Feature Correlation with Opportunity Score', fontsize=12)
axes[1].set_xlabel('Pearson r')

plt.tight_layout()
plt.savefig('../docs/screenshots/correlation_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print("Top 5 features by |correlation| with opportunity score:")
for feat, val in top5.items():
    corr_val = corr_with_score[feat]
    manual_w = {'overall_demand_gap_score':0.25,'expansion_readiness_score':0.20,
                'overall_confidence_score':0.15,'overall_risk_score':0.20,
                'pricing_power_score':0.10,'overall_margin_risk_score':0.10}.get(feat, 'N/A')
    print(f"  {feat:<35} r={corr_val:+.3f}  manual_weight={manual_w}")

## Section 3 — Sensitivity Analysis

In [ ]:
# Scoring formula from dbt_project.yml vars
WEIGHTS = {
    'overall_demand_gap_score':      0.25,
    'expansion_readiness_score':     0.20,
    'risk_inv':                      0.20,  # (100 - overall_risk_score)
    'overall_confidence_score':      0.15,
    'pricing_power_score':           0.10,
    'margin_inv':                    0.10,  # (100 - overall_margin_risk_score)
}

def compute_score(row, weights):
    s = (row['overall_demand_gap_score']      * weights['overall_demand_gap_score'] +
         row['expansion_readiness_score']     * weights['expansion_readiness_score'] +
         (100 - row['overall_risk_score'])    * weights['risk_inv'] +
         row['overall_confidence_score']      * weights['overall_confidence_score'] +
         row['pricing_power_score']           * weights['pricing_power_score'] +
         (100 - row['overall_margin_risk_score']) * weights['margin_inv'])
    return np.clip(s, 0, 100)

def score_to_tier(s):
    if s >= 75: return 'Prime'
    if s >= 55: return 'Solid'
    if s >= 35: return 'Watch'
    return 'Low'

# Verify formula reproduces existing scores
df['recomputed_score'] = df.apply(lambda r: compute_score(r, WEIGHTS), axis=1)
delta = (df['recomputed_score'] - df['overall_opportunity_score']).abs()
print(f"Formula verification — max delta from stored score: {delta.max():.2f}")
print(f"(Small delta = formula is correct; large delta = rounding differences only)")
print()

# Base tiers
df['base_tier'] = df['recomputed_score'].apply(score_to_tier)

# Run sensitivity
results_sens = []
for feat, base_w in WEIGHTS.items():
    for direction, multiplier in [('-10%', 0.9), ('+10%', 1.1)]:
        new_weights = WEIGHTS.copy()
        new_weights[feat] = base_w * multiplier
        new_scores = df.apply(lambda r: compute_score(r, new_weights), axis=1)
        new_tiers  = new_scores.apply(score_to_tier)
        changed    = (new_tiers != df['base_tier']).sum()
        pct_changed = changed / len(df) * 100
        results_sens.append({
            'Feature': feat.replace('_score','').replace('overall_','').replace('_inv',' (inv)'),
            'Base Weight': base_w,
            'Change': direction,
            'Rows Changing Tier': changed,
            '% Changing Tier': round(pct_changed, 1)
        })

sens_df = pd.DataFrame(results_sens)
print(sens_df.to_string(index=False))

In [ ]:
# Pivot for display
pivot = sens_df.pivot_table(
    index=['Feature','Base Weight'], columns='Change',
    values='% Changing Tier').reset_index()
pivot.columns.name = None
pivot = pivot.rename(columns={'-10%': 'Tier change if -10%', '+10%': 'Tier change if +10%'})
pivot['Avg sensitivity'] = pivot[['Tier change if -10%','Tier change if +10%']].mean(axis=1)
pivot = pivot.sort_values('Avg sensitivity', ascending=False)
print("Sensitivity Table (% of rows that change tier with weight adjustment):")
print(pivot.to_string(index=False))

most_sensitive  = pivot.iloc[0]['Feature']
least_sensitive = pivot.iloc[-1]['Feature']
print(f"\nMost sensitive weight:  {most_sensitive}")
print(f"Least sensitive weight: {least_sensitive}")

## Section 4 — ML Consistency Check

In [ ]:
# Binary target: score >= 65 = "strong opportunity"
THRESHOLD = 65
df['label'] = (df['overall_opportunity_score'] >= THRESHOLD).astype(int)
print(f"Binary target (score >= {THRESHOLD}):")
print(f"  Positive (1): {df['label'].sum()} rows  ({df['label'].mean()*100:.1f}%)")
print(f"  Negative (0): {(1-df['label']).sum()} rows")

X_cols = [c for c in feature_cols if c in df.columns]
train_X = train_df[X_cols].fillna(0)
test_X  = test_df[X_cols].fillna(0)
train_y = (train_df['overall_opportunity_score'] >= THRESHOLD).astype(int)
test_y  = (test_df['overall_opportunity_score'] >= THRESHOLD).astype(int)

print(f"\nTrain class balance: {train_y.mean()*100:.1f}% positive")
print(f"Test  class balance: {test_y.mean()*100:.1f}% positive")

n_unique_test = len(test_y.unique())
if n_unique_test < 2:
    print(f"\nOnly one class in test set — skipping AUC, reporting F1 and accuracy only")
    SKIP_AUC = True
else:
    SKIP_AUC = False

In [ ]:
scaler = StandardScaler()
train_Xs = scaler.fit_transform(train_X)
test_Xs  = scaler.transform(test_X)

models_ml = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, max_depth=4),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=3, random_state=42,
                              use_label_encoder=False, eval_metric='logloss', verbosity=0),
}

ml_results = []
for name, model in models_ml.items():
    if name in ['Logistic Regression']:
        model.fit(train_Xs, train_y)
        pred  = model.predict(test_Xs)
        proba = model.predict_proba(test_Xs)[:,1] if hasattr(model,'predict_proba') else pred
    else:
        model.fit(train_X, train_y)
        pred  = model.predict(test_X)
        proba = model.predict_proba(test_X)[:,1] if hasattr(model,'predict_proba') else pred

    auc = roc_auc_score(test_y, proba) if not SKIP_AUC else None
    f1  = f1_score(test_y, pred, zero_division=0)
    acc = accuracy_score(test_y, pred)
    ml_results.append({'Model': name, 'AUC': auc, 'F1': round(f1,3), 'Accuracy': round(acc,3)})

ml_df = pd.DataFrame(ml_results)
print("ML Consistency Check (IMPORTANT: 'best consistency' not 'production model')")
print(ml_df.to_string(index=False))
best_ml = ml_df.loc[ml_df['F1'].idxmax(), 'Model']
print(f"\nBest consistency model: {best_ml}")

In [ ]:
# XGBoost feature importance vs manual weights
xgb_model = models_ml['XGBoost']
fi = pd.DataFrame({
    'feature': X_cols,
    'xgb_importance': xgb_model.feature_importances_
}).sort_values('xgb_importance', ascending=False)

manual_w_map = {
    'overall_demand_gap_score': 0.25,
    'expansion_readiness_score': 0.20,
    'overall_risk_score': 0.20,
    'overall_confidence_score': 0.15,
    'pricing_power_score': 0.10,
    'overall_margin_risk_score': 0.10,
    'markdown_safety_score': 0.05,
    'price_gap_pct': 0.03,
    'adjusted_price_gap_pct': 0.02,
}
fi['manual_weight'] = fi['feature'].map(manual_w_map).fillna(0)
fi['xgb_rank']    = fi['xgb_importance'].rank(ascending=False)
fi['manual_rank'] = fi['manual_weight'].rank(ascending=False)

# Spearman rank correlation
rho, pval = spearmanr(fi['xgb_rank'], fi['manual_rank'])
print(f"Feature importance comparison (XGBoost vs manual weights):")
print(fi[['feature','xgb_importance','manual_weight','xgb_rank','manual_rank']].to_string(index=False))
print(f"\nSpearman rank correlation: r={rho:.3f}  p={pval:.3f}")
if rho > 0.6:
    print(f"XGBoost feature ranking broadly consistent with manual weights (r={rho:.2f})")
else:
    top_diff = fi[(fi['xgb_rank'] - fi['manual_rank']).abs() > 2]['feature'].tolist()
    print(f"Discrepancy — features ranked differently by ML vs manual weights: {top_diff}")

In [ ]:
# Feature importance chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fi_sorted = fi.sort_values('xgb_importance')
short_names = [f.replace('overall_','').replace('_score','').replace('_','\n') for f in fi_sorted['feature']]
y_pos = np.arange(len(fi_sorted))

axes[0].barh(y_pos, fi_sorted['xgb_importance'], color='#3B82F6', alpha=0.8)
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(short_names, fontsize=9)
axes[0].set_title('XGBoost Feature Importance', fontsize=12)
axes[0].set_xlabel('Importance')

fi_sorted2 = fi.sort_values('manual_weight')
short_names2 = [f.replace('overall_','').replace('_score','').replace('_','\n') for f in fi_sorted2['feature']]
axes[1].barh(range(len(fi_sorted2)), fi_sorted2['manual_weight'], color='#10B981', alpha=0.8)
axes[1].set_yticks(range(len(fi_sorted2)))
axes[1].set_yticklabels(short_names2, fontsize=9)
axes[1].set_title('Manual Scoring Weights', fontsize=12)
axes[1].set_xlabel('Weight')

plt.suptitle(f'XGBoost vs Manual Weights  (Spearman r={rho:.2f})', fontsize=13)
plt.tight_layout()
plt.savefig('../docs/screenshots/feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

## Section 5 — Consistency Conclusion

In [ ]:
score_std   = df['overall_opportunity_score'].std()
score_min   = df['overall_opportunity_score'].min()
score_max   = df['overall_opportunity_score'].max()
score_mean  = df['overall_opportunity_score'].mean()
low_var_flag = 'yes' if score_std < 5 else 'no'

most_sens_row  = pivot.iloc[0]
least_sens_row = pivot.iloc[-1]

auc_str = f"{ml_df.loc[ml_df['Model']==best_ml,'AUC'].values[0]:.3f}" if not SKIP_AUC else "N/A (single class)"
f1_str  = ml_df.loc[ml_df['Model']==best_ml,'F1'].values[0]

verdict = 'Consistent' if rho > 0.6 else 'Review needed'

key_finding = (f"Feature rank correlation between XGBoost and manual weights is r={rho:.2f} ({verdict}). "
               f"Most sensitive weight is '{most_sens_row['Feature']}' — "
               f"a 10% change shifts {most_sens_row['Avg sensitivity']:.1f}% of rows to a different tier.")

doc = f"""# Scoring Sensitivity and Consistency Analysis

## Purpose
Internal consistency check only. Not business outcome validation.

## Score Distribution
- Scores range: {score_min:.1f} to {score_max:.1f}
- Mean: {score_mean:.1f}, Std: {score_std:.1f}
- Low std flag: {low_var_flag}

## Sensitivity Analysis
Most sensitive weight: {most_sens_row['Feature']}
  -- {most_sens_row['Avg sensitivity']:.1f}% of rows change tier with 10% weight adjustment
Least sensitive weight: {least_sens_row['Feature']}
  -- {least_sens_row['Avg sensitivity']:.1f}% of rows change tier with 10% weight adjustment

## ML Consistency Check
Best consistency model: {best_ml}
AUC: {auc_str}  F1: {f1_str}
Feature rank correlation with manual weights: r={rho:.2f}
Verdict: {verdict}

## Key Finding
{key_finding}

## Limitations
- Internal consistency only -- not outcome validation
- 200 rows, 1 month of data, store-level holdout (5/20 test stores)
- True validation requires sales/volume data
- Rerun after 6+ months of data accumulates
"""

os.makedirs('../docs', exist_ok=True)
with open('../docs/scoring_sensitivity_analysis.md', 'w') as f:
    f.write(doc)
print(doc)
print("Saved: docs/scoring_sensitivity_analysis.md")